# 第一阶段 步骤08：从递归到循环

> 来源：《深度学习入门2：自制框架》（斋藤康毅 著，郑明智 译，人民邮电出版社 2023）
> 目标：从零构建深度学习框架 **DeZero** 的第八步。

---

## 核心目标

把步骤7 的**递归** `backward` 改成**循环**实现，提升效率，也为后续处理更复杂的计算图打基础。

## 8.1 现在的 Variable 类（递归实现的问题）

步骤7 的 `backward` 是递归结构：方法内部调用"前一个变量"的 `backward`，一路回溯到 `creator is None`。

问题：**递归每深入一层都在调用栈里堆积，嵌套很深时效率很低**。所以本步用循环替代。

In [ ]:
import numpy as np

# 承接步骤07：递归版 backward
class Variable:
    def __init__(self, data):
        self.data = data
        self.grad = None
        self.creator = None

    def set_creator(self, func):
        self.creator = func

    def backward(self):                     # 递归版（本步将替换为循环）
        f = self.creator
        if f is not None:
            x = f.input
            x.grad = f.backward(self.grad)
            x.backward()                    # 递归调用

class Function:
    def __call__(self, input):
        x = input.data
        y = self.forward(x)
        output = Variable(y)
        output.set_creator(self)
        self.input = input
        self.output = output
        return output

    def forward(self, x):
        raise NotImplementedError()

    def backward(self, gy):
        raise NotImplementedError()

class Square(Function):
    def forward(self, x):
        return x ** 2
    def backward(self, gy):
        x = self.input.data
        return 2 * x * gy

class Exp(Function):
    def forward(self, x):
        return np.exp(x)
    def backward(self, gy):
        x = self.input.data
        return np.exp(x) * gy

## 8.2 使用循环实现

思路：用一个 **`funcs` 列表**存"待处理的函数"，用 `while` 循环逐个弹出处理。

| 步骤 | 说明 |
| --- | --- |
| `funcs = [self.creator]` | 从"创造自己的函数"开始 |
| `f = funcs.pop()` | 弹出列表末尾的函数（`pop` 取出并删除） |
| `x, y = f.input, f.output` | 拿到函数的输入、输出变量 |
| `x.grad = f.backward(y.grad)` | 计算输入梯度 |
| `funcs.append(x.creator)` | 把"更前面的函数"压回列表，继续回溯 |

循环直到 `funcs` 为空（回溯到用户输入）为止。

In [ ]:
# 8.2 用循环改写 backward
def backward(self):
    funcs = [self.creator]
    while funcs:
        f = funcs.pop()                 # 取出一个待处理的函数
        x, y = f.input, f.output        # 函数的输入 / 输出
        x.grad = f.backward(y.grad)     # 计算输入梯度
        if x.creator is not None:
            funcs.append(x.creator)     # 把更前面的函数压入列表

Variable.backward = backward            # 覆盖为循环版

# 8.3 验证：结果应与递归版一致
A = Square(); B = Exp(); C = Square()
x = Variable(np.array(0.5))
a = A(x); b = B(a); y = C(b)

y.grad = np.array(1.0)
y.backward()

print(x.grad)   # 3.297442541400256（与递归版一致）

## 8.3 代码验证

循环版输出与递归版**完全一致**（`3.297442541400256`），说明改造正确。

## 这一步的"为什么"

- **为什么循环更高效？** 递归每层都占用调用栈、有函数调用开销；循环只用一个列表 + `pop`，代价小得多。
- **为什么更好扩展？** 列表可以灵活调整处理顺序——这是后续处理"分支 / 复用同一变量"等复杂计算图的基础（后面还会引入 `generation` 排序）。

---

> 预告：步骤9 让函数更好用（如 `as_array` 支持 Python 原生数值输入），步骤10 补上测试——至此第一阶段「自动微分」就完成了。